# A7 — Intelligent System Development
## Knowledge Graph + RAG + Intelligent Agent (Anaconda Jupyter)

**Mục tiêu:** minh họa chuyển dịch từ mô hình ML/DL dự đoán sang hệ thống thông minh kết hợp tri thức có cấu trúc, truy hồi tài liệu và agent sử dụng công cụ.

> Notebook được thiết kế chạy từ thư mục dự án. Bản demo dùng TF-IDF + NetworkX, không cần API key. Các kết quả ở dưới phải được tạo bằng cách chạy cell thực tế; chụp màn hình sau khi chạy để làm minh chứng.

In [ ]:
# Nếu thiếu thư viện, bỏ dấu # và chạy một lần:
# %pip install -r requirements.txt

import json, sys
from pathlib import Path
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT / "A7_Intelligent_System" / "data").exists():
    ROOT = ROOT / "A7_Intelligent_System"
print("Project root:", ROOT.resolve())
print("Python:", sys.version.split()[0])
print("Core imports OK")

## 1. Khảo sát dữ liệu và triples Knowledge Graph

In [ ]:
documents = json.loads((ROOT / "data" / "knowledge_base.json").read_text(encoding="utf-8"))
triples = json.loads((ROOT / "data" / "knowledge_graph.json").read_text(encoding="utf-8"))
print("Documents:", len(documents))
print("Triples:", len(triples))
pd.DataFrame(triples).head(10)

In [ ]:
G = nx.MultiDiGraph()
for t in triples:
    G.add_edge(t["subject"], t["object"], relation=t["relation"])
print(f"Nodes = {G.number_of_nodes()}, edges = {G.number_of_edges()}")
for s, o, d in G.edges(data=True):
    print(f"{s} --{d['relation']}--> {o}")

## 2. Chunking và truy hồi TF-IDF

In [ ]:
def chunk_text(text, chunk_size=90, overlap=18):
    words = text.split()
    if not words: return []
    step = max(1, chunk_size - min(overlap, chunk_size - 1))
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), step)]

chunks = []
for doc in documents:
    for j, txt in enumerate(chunk_text(doc["text"])):
        chunks.append({"text": txt, "source": doc["source"], "topic": doc["topic"], "chunk_id": f"{doc['topic']}_{j}"})

vectorizer = TfidfVectorizer(ngram_range=(1,2), strip_accents="unicode")
X = vectorizer.fit_transform([c["text"] for c in chunks])
print("Chunks:", len(chunks), "| TF-IDF matrix:", X.shape)

def retrieve(query, top_k=3):
    q = vectorizer.transform([query])
    scores = cosine_similarity(q, X).ravel()
    ranked = np.argsort(scores)[::-1][:top_k]
    return [{**chunks[i], "score": float(scores[i])} for i in ranked if scores[i] > 0]

question = "How does RAG retrieve information before generating an answer?"
results = retrieve(question, top_k=3)
for r in results:
    print(f"score={r['score']:.3f} | topic={r['topic']} | source={r['source']}\n{r['text']}\n")

## 3. Truy vấn Knowledge Graph

In [ ]:
def graph_facts(entity):
    facts = []
    if entity not in G:
        return facts
    for _, target, data in G.out_edges(entity, data=True):
        facts.append(f"{entity} --{data['relation']}--> {target}")
    for source, _, data in G.in_edges(entity, data=True):
        facts.append(f"{source} --{data['relation']}--> {entity}")
    return facts

graph_facts("RAG")

## 4. Kết hợp KG + RAG thành câu trả lời có bằng chứng

In [ ]:
def hybrid_answer(query, top_k=3):
    retrieved = retrieve(query, top_k)
    relevant_entities = [n for n in G.nodes if n.lower() in query.lower()]
    facts = []
    for ent in relevant_entities:
        facts.extend(graph_facts(ent))
    if not retrieved and not facts:
        answer = "Chưa tìm thấy bằng chứng phù hợp trong kho tri thức hiện tại."
    else:
        answer = "Dựa trên kho tri thức hiện tại:\n"
        if facts:
            answer += "\nKnowledge Graph:\n" + "\n".join("- " + f for f in dict.fromkeys(facts))
        if retrieved:
            answer += "\n\nBằng chứng tài liệu:\n" + "\n".join(f"- [{r['topic']} | {r['source']}] {r['text']}" for r in retrieved)
        answer += "\n\nĐây là bản minh họa tổng hợp theo bằng chứng, chưa sử dụng LLM sinh tự do."
    return answer, retrieved, list(dict.fromkeys(facts))

q = "What is RAG and what does it use?"
answer, retrieved, facts = hybrid_answer(q)
print(answer)

## 5. Intelligent Agent: chọn công cụ theo mục tiêu

In [ ]:
def agent(query, top_k=3):
    q = query.lower()
    plan = []
    if any(k in q for k in ["graph", "relationship", "relation", "knowledge graph", "kg", "đồ thị", "quan hệ"]):
        plan.append("Knowledge Graph query")
    if any(k in q for k in ["what", "why", "how", "explain", "rag", "document", "tài liệu", "là gì", "tại sao"]):
        plan.append("RAG document retrieval")
    if not plan:
        plan = ["RAG document retrieval", "Knowledge Graph query"]
    answer, docs, facts = hybrid_answer(query, top_k)
    return {"plan": plan, "answer": answer, "docs": docs, "facts": facts}

agent_result = agent("Explain how RAG works and show its relationships in the Knowledge Graph")
print("Agent plan:", " -> ".join(agent_result["plan"]))
print(agent_result["answer"])

## 6. Đánh giá nhanh trên bộ câu hỏi

In [ ]:
eval_df = pd.read_csv(ROOT / "evaluation" / "questions.csv")
rows = []
for _, row in eval_df.iterrows():
    retrieved = retrieve(row["question"], top_k=3)
    topics = [x["topic"] for x in retrieved]
    hit = row["expected_topic"] in topics
    rows.append({
        "question": row["question"],
        "expected_topic": row["expected_topic"],
        "retrieved_topics": ", ".join(topics),
        "hit_at_3": hit
    })
eval_results = pd.DataFrame(rows)
display(eval_results)
print(f"Hit@3 = {eval_results['hit_at_3'].mean():.1%} ({eval_results['hit_at_3'].sum()}/{len(eval_results)})")

## 7. Kết luận thực nghiệm và chụp minh chứng

- **Knowledge Graph** biểu diễn quan hệ tường minh bằng triples.
- **RAG retrieval** chọn các đoạn tài liệu có độ tương đồng TF-IDF cao.
- **Hybrid** kết hợp facts từ graph và bằng chứng từ tài liệu.
- **Agent** chọn công cụ theo luật; đây là agent minh bạch, chưa phải tác tử tự chủ dựa trên LLM.
- **Đánh giá** Hit@3 chỉ là kiểm tra nhỏ trên bộ câu hỏi mẫu; cần thêm câu hỏi, kiểm tra Recall@k, chất lượng câu trả lời, tính groundedness và lỗi.

Chụp màn hình các cell: import thành công; số node/cạnh; top-k với score và source; câu trả lời hybrid; agent plan; bảng đánh giá. Lưu ảnh thật vào thư mục `screenshots/`. Không điền số liệu nếu chưa chạy notebook.